In [233]:
from netgen.occ import *
from netgen.webgui import Draw as DrawGeo
import ngsolve

In [234]:
pfeiler11=Box((0,0,0), (1,1,30))
pfeiler12=Box((11,0,0),(12,1,30))
strebe11=Box((1,0.2,15),(11,0.8,18))
strebe12=Box((1,0.2,26),(11,0.8,29))

pfeiler21=Box((0,30,0), (1,1+30,30))
pfeiler22=Box((11,30,0),(12,1+30,30))
strebe21=Box((1,0.2+30,15),(11,0.8+30,18))
strebe22=Box((1,0.2+30,26),(11,0.8+30,29))
road1=Box((1,1,10),(11,-5,11))
road2=Box((1,30,10),(11,35,11))

road=Box((1,-5,10),(11,35,11))

strebe11.faces.name="fix"
strebe12.faces.name="fix"
strebe21.faces.name="fix"
strebe22.faces.name="fix"

pfeiler11.faces.name="fix"
pfeiler12.faces.name="fix"
pfeiler21.faces.name="fix"
pfeiler22.faces.name="fix"

road.faces.Min(Y).name = "fix"
road.faces.Max(Y).name = "fix"
road.faces.Max(X).name = "force"
road.faces.Max(Z).name = "force"

geo=pfeiler11+pfeiler12+strebe11+strebe12
geo+=pfeiler21+pfeiler22+strebe21+strebe22
geo+=road

DrawGeo(geo, height="3vh")

WebGuiWidget(layout=Layout(height='3vh', width='100%'), value={'ngsolve_version': 'Netgen x.x', 'mesh_dim': 3,…

BaseWebGuiScene

In [235]:
from ngsolve import *
from ngsolve.webgui import Draw
mesh = Mesh(OCCGeometry(geo).GenerateMesh(maxh=1)).Curve(3)
Draw (mesh, height="3vh");

WebGuiWidget(layout=Layout(height='3vh', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.240…

In [236]:
E, nu = 210, 0.2
mu  = E / 2 / (1+nu)
lam = E * nu / ((1+nu)*(1-2*nu))

def Stress(strain):
    return 2*mu*strain + lam*Trace(strain)*Id(3)

In [237]:
fes = VectorH1(mesh, order=3, dirichlet="fix")
u,v = fes.TnT()
gfu = GridFunction(fes)

with TaskManager():
    a = BilinearForm(InnerProduct(Stress(Sym(Grad(u))), Sym(Grad(v))).Compile()*dx)
    pre = Preconditioner(a, "bddc")
    a.Assemble()

In [238]:
force = CF( (2e-5,2e-5,0) )
f = LinearForm(force*v*ds("force")).Assemble()

In [239]:
from ngsolve.krylovspace import CGSolver
inv = CGSolver(a.mat, pre, printrates=True, tol=1e-8)
gfu.vec.data = inv * f.vec

CG iteration 1, residual = 0.0009690159210794601     
CG iteration 2, residual = 0.0006227811507850242     
CG iteration 3, residual = 0.0005350961257663297     
CG iteration 4, residual = 0.001225658914060247     
CG iteration 5, residual = 0.0010734392853149815     
CG iteration 6, residual = 0.000302095854867781     
CG iteration 7, residual = 0.0002743309532697311     
CG iteration 8, residual = 0.00015272956241924217     
CG iteration 9, residual = 0.0001447044359440352     
CG iteration 10, residual = 8.539183723462933e-05     
CG iteration 11, residual = 6.927784850456013e-05     
CG iteration 12, residual = 3.784288033200765e-05     
CG iteration 13, residual = 2.2803964594689715e-05     
CG iteration 14, residual = 1.6330402858304527e-05     
CG iteration 15, residual = 4.3619771515951044e-05     
CG iteration 16, residual = 1.3630604190439541e-05     
CG iteration 17, residual = 1.0381664543483272e-05     
CG iteration 18, residual = 6.451165054594933e-06     
CG iteration 19

In [240]:
with TaskManager():
    fesstress = MatrixValued(H1(mesh,order=3), symmetric=True)
    gfstress = GridFunction(fesstress)
    gfstress.Interpolate (Stress(Sym(Grad(gfu))))

In [241]:
Draw (5e4*gfu, mesh, height="3vh");

WebGuiWidget(layout=Layout(height='3vh', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.240…

In [242]:
Draw (Norm(gfstress), mesh, deformation=1e4*gfu, draw_vol=False, order=3, height="3vh");

WebGuiWidget(layout=Layout(height='3vh', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.240…